In [2]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import LeftRightImagery
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import accuracy_score, classification_report

paradigm = LeftRightImagery()
dataset = BNCI2014_001()

In [23]:
x, y, metadata = paradigm.get_data(dataset, subjects=[1])

x_train = x[0:144]
y_train = y[0:144]
x_test = x[0:144]
y_test = y[0:144]

ch_names = [
    "Fz", "FC3", "FC1", "FCz", "FC2", "FC4",
    "C5", "C3", "C1", "Cz", "C2", "C4", "C6",
    "CP3", "CP1", "CPz", "CP2", "CP4",
    "P1", "Pz", "P2", "POz"
]

In [ ]:
from mne.time_frequency import psd_array_welch

sfreq = 250  # BNCI2014_001 sampling frequency

# Compute PSD for all epochs — returns (n_epochs, n_channels, n_freqs)
psds, freqs = psd_array_welch(x, sfreq=sfreq, fmin=1, fmax=40, n_fft=256)

print(f"PSDs shape: {psds.shape}")   # (288, 22, n_freqs)
print(f"Frequency bins: {freqs}")


Effective window size : 1.024 (s)
PSDs shape: (288, 22, 39)
Frequency bins: [ 1.953125   2.9296875  3.90625    4.8828125  5.859375   6.8359375
  7.8125     8.7890625  9.765625  10.7421875 11.71875   12.6953125
 13.671875  14.6484375 15.625     16.6015625 17.578125  18.5546875
 19.53125   20.5078125 21.484375  22.4609375 23.4375    24.4140625
 25.390625  26.3671875 27.34375   28.3203125 29.296875  30.2734375
 31.25      32.2265625 33.203125  34.1796875 35.15625   36.1328125
 37.109375  38.0859375 39.0625   ]


In [24]:
# Frequency bands
theta_band = (4, 8)   # fatigue / drowsiness
alpha_band = (8, 13)  # idle / disengagement
beta_band  = (13, 30) # engagement / active processing

theta_mask = (freqs >= theta_band[0]) & (freqs <= theta_band[1])
alpha_mask = (freqs >= alpha_band[0]) & (freqs <= alpha_band[1])
beta_mask  = (freqs >= beta_band[0]) & (freqs <= beta_band[1])

# Frontal for stress: Fz, FC1, FC2
stress_channels = ["Fz", "FC1", "FC2"]
stress_idx = [ch_names.index(ch) for ch in stress_channels]

# Parietal/occipital for fatigue: Pz, POz
fatigue_channels = ["Pz", "POz"]
fatigue_idx = [ch_names.index(ch) for ch in fatigue_channels]

# Central for engagement: C3, Cz, C4
engage_channels = ["C3", "Cz", "C4"]
engage_idx = [ch_names.index(ch) for ch in engage_channels]

In [25]:
# Average PSD across selected channels, then across frequency bins
theta_power = psds[:, fatigue_idx][:, :, theta_mask].mean(axis=(1, 2))
alpha_power = psds[:, fatigue_idx][:, :, alpha_mask].mean(axis=(1, 2))
beta_power  = psds[:, engage_idx][:, :, beta_mask].mean(axis=(1, 2))

# Optional engagement index
engagement_index = beta_power / (alpha_power + theta_power + 1e-8)

In [31]:
fatigue = theta_power / beta_power

In [32]:
print("Training Trials")
print(fatigue[0])
print(fatigue[70])
print(fatigue[140])

print("Testing Trials")
print(fatigue[144])
print(fatigue[214])
print(fatigue[287])

print(f"{fatigue[214]} vs {fatigue[287]}")

Training Trials
0.08032523753037617
0.26014517445677665
0.822183243864192
Testing Trials
0.08851932812360368
0.25926681388127437
0.1285724768109312
0.25926681388127437 vs 0.1285724768109312


In [30]:
print("Training Trials")
print(engagement_index[0])
print(engagement_index[70])
print(engagement_index[140])

print("Testing Trials")
print(engagement_index[144])
print(engagement_index[214])
print(engagement_index[287])

print(f"{engagement_index[214]} vs {engagement_index[287]}")

Training Trials
0.30214355064806303
0.16687385155601875
0.1317679760302363
Testing Trials
0.5598993254966546
0.21137809776451924
0.4174687839179203
0.21137809776451924 vs 0.4174687839179203


In [4]:
metadata

,subject,session,run
0,1,0train,0
1,1,0train,0
2,1,0train,0
3,1,0train,0
4,1,0train,0
...,...,...,...
283,1,1test,5
284,1,1test,5
285,1,1test,5
286,1,1test,5


In [5]:
csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)

# x_csp_train = csp.fit_transform(x (i.e full data instead of [0:144]), y)   # DATA  LEAKAGE ERROR, identified patterns of future data also, unreliable results
x_csp_train = csp.fit_transform(x_train, y_train)

ld = LDA()
ld.fit(x_csp_train,y_train)
x_csp_test = csp.transform(x_test) # Why not fit trasnform here? it would be like having answer key while solving, it would learn the exact pattenrs it is trying to predict
y_pred = ld.predict(x_csp_test)

# accuracy_score(y_test, y_pred)
print(classification_report(y_test, y_pred))

Computing rank from data with rank=None
    Using tolerance 36 (2.2e-16 eps * 22 dim * 7.4e+15  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=left_hand covariance using EMPIRICAL
Done.
Estimating class=right_hand covariance using EMPIRICAL
Done.
              precision    recall  f1-score   support

   left_hand       0.93      0.88      0.90        72
  right_hand       0.88      0.93      0.91        72

    accuracy                           0.90       144
   macro avg       0.90      0.90      0.90       144
weighted avg       0.90      0.90      0.90       144

